In [0]:
%sql
create catalog if not exists dev;

In [0]:
#Paqueterías
import ast
from pyspark.sql.functions import col, udf, explode
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, LongType

In [0]:
# Lectura (Con el separador '|')
file_path = "/Volumes/dev/ciencias_data/session_data/sessions_part1.csv"

df_raw = spark.read.csv(
    file_path,
    header=True,
    sep="|", 
    inferSchema=True,
    quote='"',
    escape='"'
)

df_raw.show()

In [0]:
# Definición de Esquema
element_schema = StructType([
    StructField("srcDataBytes", LongType(), True),
    StructField("dstBytes", LongType(), True),
    StructField("packetLen", ArrayType(LongType()), True),
    StructField("srcPort", LongType(), True),
    StructField("totPackets", LongType(), True),
    StructField("packetPos", ArrayType(LongType()), True),
    StructField("srcPayload", StringType(), True),
    StructField("segmentCnt", LongType(), True),
    StructField("srcPackets", LongType(), True),
    StructField("protocol", StringType(), True),
    StructField("lastPacket", LongType(), True),
    StructField("dstPort", LongType(), True),
    StructField("communityId", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("srcASN", StringType(), True),
    StructField("dstASN", StringType(), True)
])


output_schema = ArrayType(element_schema)

In [0]:
def parse_safe(data_str):
    """
    Intenta parsear la cadena. Si falla por CUALQUIER razón, 
    devuelve una lista vacía para no romper el proceso.
    """
    if data_str is None:
        return []
    
    try:
        # Intentamos evaluar la estructura
        parsed = ast.literal_eval(data_str)
        
        # Verificamos que sea una lista (es lo que espera el esquema)
        if isinstance(parsed, list):
            return parsed
        elif isinstance(parsed, dict):
            return [parsed] # Si es un solo dict, lo metemos en una lista
        return []
        
    except Exception as e:
        # Capturamos TODO (Exception). 
        # Si la cadena está cortada, tiene caracteres raros o el delimitador rompió el string,
        # simplemente devolvemos lista vacía y seguimos con la siguiente fila.
        return []

In [0]:
# Registramos la UDF segura
parse_udf_safe = udf(parse_safe, output_schema)

# Procesamiento

# Aplicamos la UDF
df_parsed = df_raw.withColumn("data_parsed", parse_udf_safe(col("data")))

# Filtramos las filas que no se pudieron leer (opcional, para limpieza)
# df_parsed = df_parsed.filter(size(col("data_parsed")) > 0)

# Explode y selección
df_exploded = df_parsed.select(explode(col("data_parsed")).alias("col"))
df_final = df_exploded.select("col.*")

print("Mostrando resultados procesados:")
df_final.display()